Description :

This notebook preprocesses the raw extracted text by cleaning and segmenting it into meaningful chunks. Proper text cleaning and chunking are essential to improve retrieval accuracy and reduce noise in the vector search stage of the RAG pipeline.

Key tasks performed:

Remove unnecessary whitespace and formatting artifacts

Normalize text for consistency

Split large documents into overlapping text chunks suitable for embeddings

In [1]:
import json

with open("Data/railway_rules_raw.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(len(data))

<class 'list'>
4


In [2]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'Page \d+', '', text)
    return text.strip()

for doc in data:
    doc["content"] = clean_text(doc["content"])

print("Cleaning done")

Cleaning done


In [3]:
def chunk_text(text, chunk_size=350, overlap=50):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.strip()) > 50:
            chunks.append(chunk)

    return chunks

In [4]:
chunked_data = []

for doc in data:
    chunks = chunk_text(doc["content"])
    for idx, chunk in enumerate(chunks):
        chunked_data.append({
            "source": doc["source"],
            "chunk_id": idx,
            "text": chunk
        })

print("Total chunks:", len(chunked_data))

Total chunks: 23


In [5]:
with open("Data/railway_rules_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunked_data, f, ensure_ascii=False, indent=2)

print("Chunked dataset saved")

Chunked dataset saved


In [6]:
chunked_data[0]

{'source': 'CitizenCharter.pdf',
 'chunk_id': 0,
 'text': '322 Citizens’ Charter on Passenger Services of Indian Railways Preamble This Charter is a commitment of the Indian Railway Administration to: • Provide safe and dependable train services • Set notified standards for various services, wherever possible • Ensure adequate passenger amenities in trains and at railway stations; • Provide courteous and efficient counter services; and • Set up a responsive and effective Grievance Redressal Machinery, at various levels and time-bound resolution of complaints and grievances as far as possible. Reservations • Provision of computerised reservation facilities at all stations with a workload of 100 reservation related transactions. • Opening of adequate number of counters to ensure reduced waiting time. Booking • Opening of ticket booking counters with adequate working hours to facilitate issue of tickets to the public. The working hours will be clearly displayed at the counters. Refunds • 